In [1]:
# https://learnopencv.com/fine-tuning-bert/

In [2]:
from pathlib import Path

import torch

from collections.abc import Callable

from datasets import Dataset, load_dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

import evaluate
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from pathlib import Path


import sys

sys.path.insert(0, str(Path.cwd().parent))


from finetuning.commons import PipelineData, prepare_data, parse_pubtator, build_training_samples, samples_to_rels_like_df
from dataset_preparation.perturbations import BIORED_RELATION_TYPES, NO_RELATION_LABEL

In [3]:
USE_CACHE = False

In [4]:
from datetime import datetime
OUT_DIR = f"relations-bert-{datetime.now():%Y-%m-%d-%H-%M-%S}"

# NOTE: these are the defaults might change according to avaibale VRAM
# -> BATCH SIZE and LR are halved if less than 8GB of VRAM is detected
BATCH_SIZE = 32
NUM_PROCS = 32
LR = 0.00005
EPOCHS = 10
MODEL = 'NeuML/pubmedbert-base-embeddings'
CACHE_DIR = Path("cache")

PUBTATOR_FILE = Path.home() / "git/LLMs_for_NEL/Data/BioRED/Dev.PubTator"



def load_or_cache(split: str, prefix: str, build: Callable[[], Dataset], use_cache: bool = True) -> Dataset:
    """Load a Hugging Face ``Dataset`` from disk cache or build and persist it.

    Cached data is stored under ``CACHE_DIR / f"{prefix}_{split}"`` (default: ``cache/``).
    On a cache hit, ``load_from_disk`` is used and ``build`` is not called.
    On a miss, ``build()`` runs once, the result is saved with ``save_to_disk``, then returned.

    Args:
        split: Split identifier used in the cache directory name (e.g. ``"train"``, ``"validation"``).
        prefix: Stage prefix distinguishing pipeline steps (e.g. ``"raw"``, ``"tokenized"``).
        build: Zero-argument callable that produces the dataset when the cache is missing.

    Returns:
        The dataset for the given split, either loaded from cache or freshly built.

    Examples:
        Download a Hub split and cache it as ``cache/raw_train/``::

            train = load_or_cache(
                "train",
                "raw",
                lambda: load_dataset("ccdv/arxiv-classification", split="train"),
            )

        Tokenize an in-memory split and cache as ``cache/tokenized_train/``::

            tokenized_train = load_or_cache(
                "train",
                "tokenized",
                lambda: train.map(preprocess_function, batched=True, batch_size=32),
            )

    Note:
        Delete the matching folder under ``cache/`` to force a rebuild after changing
        ``build``, the source data, or preprocessing.
    """
    cache_path = CACHE_DIR / f"{prefix}_{split}"
    if cache_path.exists() and use_cache:
        print(f"Loading {prefix} {split} from cache/")
        return load_from_disk(cache_path)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    dataset = build()
    dataset.save_to_disk(cache_path)
    return dataset


In [5]:
MODE: str = "cpu"
if torch.backends.mps.is_available():
    MODE = "mps"
elif torch.cuda.is_available():
    MODE = "cuda"
else:
    print("No GPU or MPS available - uising CPU")

print(f"Using {MODE} for training")


hardware_specific_args = {}

if MODE == "mps":
    hardware_specific_args["fp16"] = False
    hardware_specific_args["dataloader_num_workers"] = 0
    hardware_specific_args["per_device_train_batch_size"] = BATCH_SIZE
    hardware_specific_args["per_device_eval_batch_size"] = BATCH_SIZE
    hardware_specific_args["learning_rate"] = LR
    

elif MODE == "cuda":
    # Check total GPU VRAM 
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA device VRAM: {total_vram_gb:.2f} GB")

    # Default: assume >8GB VRAM
    batch_div = 1
    lr_div = 1

    # 2070super only has 8gigs of VRAM :')
    if total_vram_gb <= 8.5:
        print("Detected ~8GB of VRAM or less, reducing batch size and learning rate.")
        batch_div = 2
        lr_div = 2

    hardware_specific_args["fp16"] = True
    hardware_specific_args["per_device_train_batch_size"] = BATCH_SIZE // batch_div
    hardware_specific_args["per_device_eval_batch_size"] = BATCH_SIZE // batch_div
    hardware_specific_args["learning_rate"] = LR / lr_div

    print(f"Using {hardware_specific_args['per_device_train_batch_size']} for training")
    print(f"Using {hardware_specific_args['per_device_eval_batch_size']} for evaluation")
    print(f"Using {hardware_specific_args['learning_rate']} for learning rate")



Using cuda for training
CUDA device VRAM: 7.57 GB
Detected ~8GB of VRAM or less, reducing batch size and learning rate.
Using 16 for training
Using 16 for evaluation
Using 2.5e-05 for learning rate


In [6]:
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    save_total_limit=EPOCHS,
    report_to="tensorboard",
    **hardware_specific_args,
)

In [7]:
RELATION_LABELS: list[str] = BIORED_RELATION_TYPES + [NO_RELATION_LABEL]
label2id: dict[str, int] = {name: idx for idx, name in enumerate(RELATION_LABELS)}
id2label: dict[int, str] = {idx: name for name, idx in label2id.items()}
MASK_TOKEN = "[MASK]"

meta_df, anns_df, rels_df = parse_pubtator(PUBTATOR_FILE)

samples = build_training_samples(meta_df, anns_df, rels_df)
samples = samples_to_rels_like_df(samples)

# Gold relations (8 types) + unrelated entity pairs (NoRelation).
samples = samples[samples["perturbation"].isin(["gold", "false_positive"])].copy()

samples["prompt"] = samples.apply(
    lambda row: (
        f"Relation: {row['entity_a_text']} -> {MASK_TOKEN} -> {row['entity_b_text']}\n"
        f"Context: {row['abstract']}"
    ),
    axis=1,
)
samples["target_relation"] = np.where(
    samples["perturbation"] == "false_positive",
    NO_RELATION_LABEL,
    samples["relation_type"],
)
samples["label"] = samples["target_relation"].map(label2id)

train_df, val_df = train_test_split(
    samples,
    test_size=0.1,
    random_state=42,
    stratify=samples["label"],
)
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
valid_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
print(f"Classes: {len(RELATION_LABELS)}")
print(samples["target_relation"].value_counts())

{'pmid': '14510914', 'relation_type': 'Positive_Correlation', 'id_1': 'c|DEL|1314_1328|', 'id_2': 'D003409', 'entity_a_text': 'deletion of the coding sequence (nt 1314 through nt 1328)', 'entity_b_text': 'Congenital hypothyroidism', 'perturbation': 'gold', 'label': 1, 'abstract': "OBJECTIVE: Iodide transport defect (ITD) is a rare disorder characterised by an inability of the thyroid to maintain an iodide gradient across the basolateral membrane of thyroid follicular cells, that often results in congenital hypothyroidism. When present the defect is also found in the salivary glands and gastric mucosa and it has been shown to arise from abnormalities of the sodium/iodide symporter (NIS). PATIENT: We describe a woman with hypothyroidism identified at the 3rd month of life. The diagnosis of ITD was suspected because of nodular goitre, and little if any iodide uptake by the thyroid and salivary glands. Treatment with iodide partially corrected the hypothyroidism; however, long-term substit

In [8]:
train_dataset[0]

{'pmid': '25119790',
 'relation_type': 'Positive_Correlation',
 'id_1': 'C030272',
 'id_2': 'D017382',
 'entity_a_text': 'maleate',
 'entity_b_text': 'reactive oxygen species',
 'perturbation': 'gold',
 'label': 0,
 'abstract': 'The potential protective effect of the dietary antioxidant curcumin (120 mg/Kg/day for 6 days) against the renal injury induced by maleate was evaluated. Tubular proteinuria and oxidative stress were induced by a single injection of maleate (400 mg/kg) in rats. Maleate-induced renal injury included increase in renal vascular resistance and in the urinary excretion of total protein, glucose, sodium, neutrophil gelatinase-associated lipocalin (NGAL) and N-acetyl b-D-glucosaminidase (NAG), upregulation of kidney injury molecule (KIM)-1, decrease in renal blood flow and claudin-2 expression besides of necrosis and apoptosis of tubular cells on 24 h. Oxidative stress was determined by measuring the oxidation of lipids and proteins and diminution in renal Nrf2 levels

In [9]:
print(f"Relation classification: {len(RELATION_LABELS)} classes")
print("label2id:", label2id)

Relation classification: 9 classes
label2id: {'Positive_Correlation': 0, 'Negative_Correlation': 1, 'Association': 2, 'Comparison': 3, 'Cotreatment': 4, 'Drug_Interaction': 5, 'Bind': 6, 'Conversion': 7, 'NoRelation': 8}


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [11]:
def preprocess_function(examples, key: str = "prompt"):
    return tokenizer(
        examples[key],
        truncation=True,
        padding=True,
        max_length=512,
    )

In [12]:
def _tokenize(dataset: Dataset) -> Dataset:
    return dataset.map(
        preprocess_function,
        batched=True,
        batch_size=BATCH_SIZE,
        num_proc=NUM_PROCS,
    )


tokenized_train = load_or_cache(
    "train", "bio_red_relation_type_tokenized", lambda: _tokenize(train_dataset), use_cache=USE_CACHE
)
tokenized_valid = load_or_cache(
    "valid", "bio_red_relation_type_tokenized", lambda: _tokenize(valid_dataset), use_cache=USE_CACHE
)
# tokenized_test = load_or_cache("test", "bio_red_relation_type_tokenized", lambda: _tokenize(test_dataset))

Parameter 'function'=<function preprocess_function at 0x7ff45e607ba0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map (num_proc=32):   0%|          | 0/2004 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2004 [00:00<?, ? examples/s]

Map (num_proc=32):   0%|          | 0/223 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/223 [00:00<?, ? examples/s]

In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [14]:
tokenized_sample = preprocess_function(train_dataset[0])
print(tokenized_sample)
print(f"Length of tokenized IDs: {len(tokenized_sample.input_ids)}")
print(f"Length of attention mask: {len(tokenized_sample.attention_mask)}")

{'input_ids': [2, 6147, 30, 4005, 2010, 17, 34, 4, 17, 34, 6079, 4612, 3038, 4491, 30, 1920, 2889, 6337, 2495, 1927, 1920, 6096, 7335, 12960, 12, 6103, 2786, 19, 3621, 19, 3007, 1958, 26, 3000, 13, 3316, 1920, 4604, 4047, 2719, 2007, 4005, 2010, 1982, 3747, 18, 11755, 15831, 1930, 5163, 3071, 1985, 2719, 2007, 43, 2957, 4107, 1927, 4005, 2010, 12, 5805, 2786, 19, 3621, 13, 1922, 3615, 18, 4005, 2010, 17, 2719, 4604, 4047, 3063, 2760, 1922, 4604, 4541, 3554, 1930, 1922, 1920, 7075, 11093, 1927, 2698, 2213, 16, 3817, 16, 5227, 16, 9460, 13200, 2051, 17, 2458, 3178, 5000, 1921, 12, 28768, 13, 1930, 56, 17, 6894, 44, 17, 46, 17, 3351, 5360, 8547, 9200, 12, 14466, 13, 16, 8346, 1927, 5245, 4047, 5388, 12, 10453, 13, 17, 21, 16, 3714, 1922, 4604, 2877, 3426, 1930, 19609, 17, 22, 2294, 8820, 1927, 7476, 1930, 4091, 1927, 11755, 2094, 1990, 2686, 50, 18, 5163, 3071, 1982, 3180, 2007, 6711, 1920, 6638, 1927, 8238, 1930, 2697, 1930, 7798, 2306, 1922, 4604, 10933, 2428, 18, 2351, 1985, 2222, 3609

In [15]:
tokenized_sample = preprocess_function(train_dataset[0])
print(tokenized_sample)

{'input_ids': [2, 6147, 30, 4005, 2010, 17, 34, 4, 17, 34, 6079, 4612, 3038, 4491, 30, 1920, 2889, 6337, 2495, 1927, 1920, 6096, 7335, 12960, 12, 6103, 2786, 19, 3621, 19, 3007, 1958, 26, 3000, 13, 3316, 1920, 4604, 4047, 2719, 2007, 4005, 2010, 1982, 3747, 18, 11755, 15831, 1930, 5163, 3071, 1985, 2719, 2007, 43, 2957, 4107, 1927, 4005, 2010, 12, 5805, 2786, 19, 3621, 13, 1922, 3615, 18, 4005, 2010, 17, 2719, 4604, 4047, 3063, 2760, 1922, 4604, 4541, 3554, 1930, 1922, 1920, 7075, 11093, 1927, 2698, 2213, 16, 3817, 16, 5227, 16, 9460, 13200, 2051, 17, 2458, 3178, 5000, 1921, 12, 28768, 13, 1930, 56, 17, 6894, 44, 17, 46, 17, 3351, 5360, 8547, 9200, 12, 14466, 13, 16, 8346, 1927, 5245, 4047, 5388, 12, 10453, 13, 17, 21, 16, 3714, 1922, 4604, 2877, 3426, 1930, 19609, 17, 22, 2294, 8820, 1927, 7476, 1930, 4091, 1927, 11755, 2094, 1990, 2686, 50, 18, 5163, 3071, 1982, 3180, 2007, 6711, 1920, 6638, 1927, 8238, 1930, 2697, 1930, 7798, 2306, 1922, 4604, 10933, 2428, 18, 2351, 1985, 2222, 3609

In [16]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    """Compute accuracy and weighted F1 for 9-way relation-type classification."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        **accuracy_metric.compute(predictions=predictions, references=labels),
        **f1_metric.compute(predictions=predictions, references=labels, average="weighted"),
    }

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=len(RELATION_LABELS),
    id2label=id2label,
    label2id=label2id,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at NeuML/pubmedbert-base-embeddings and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
if MODE != "cpu":
    model = model.to(MODE)
print(f"Moving model to {MODE}")


Moving model to cuda


In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [21]:
history = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.232446,0.488789,0.380820
2,No log,0.884307,0.663677,0.665028
3,No log,0.813766,0.704036,0.703799
4,0.931600,0.755641,0.735426,0.726657
5,0.931600,0.850064,0.717489,0.704232
6,0.931600,0.688466,0.762332,0.758183
7,0.931600,0.742039,0.793722,0.786961
8,0.323700,0.776783,0.775785,0.764126
9,0.323700,0.708009,0.789238,0.783937
10,0.323700,0.727538,0.798206,0.790604


In [22]:
eval_results = trainer.evaluate(tokenized_valid)
print(eval_results)

predictions = trainer.predict(tokenized_valid)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        pred_labels,
        labels=list(range(len(RELATION_LABELS))),
        target_names=RELATION_LABELS,
        digits=3,
        zero_division=0,
    )
)

{'eval_loss': 0.7275375127792358, 'eval_accuracy': 0.7982062780269058, 'eval_f1': 0.7906044548389858, 'eval_runtime': 1.6109, 'eval_samples_per_second': 138.43, 'eval_steps_per_second': 8.691, 'epoch': 10.0}
                      precision    recall  f1-score   support

Positive_Correlation      0.757     0.824     0.789        34
Negative_Correlation      0.522     0.571     0.545        21
         Association      0.778     0.648     0.707        54
          Comparison      0.000     0.000     0.000         1
         Cotreatment      0.000     0.000     0.000         1
    Drug_Interaction      0.000     0.000     0.000         0
                Bind      0.000     0.000     0.000         1
          Conversion      0.000     0.000     0.000         0
          NoRelation      0.873     0.928     0.900       111

            accuracy                          0.798       223
           macro avg      0.325     0.330     0.327       223
        weighted avg      0.787     0.798     

In [23]:
model.save_pretrained(f"{OUT_DIR}_dump")  # TODO

In [26]:
OUT_DIR

'relations-bert-2026-06-02-20-58-25'

In [24]:
	
trainer.evaluate(tokenized_valid)

{'eval_loss': 0.7275375127792358,
 'eval_accuracy': 0.7982062780269058,
 'eval_f1': 0.7906044548389858,
 'eval_runtime': 1.5948,
 'eval_samples_per_second': 139.828,
 'eval_steps_per_second': 8.778,
 'epoch': 10.0}

In [27]:
# Load a trained checkpoint (must be 9-class relation-type model, not the old binary one).
CHECKPOINT_DIR = Path(OUT_DIR) / "checkpoint-1260"

model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

if MODE != "cpu":
    model = model.to(MODE)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

eval_results = trainer.evaluate(tokenized_valid)
print(eval_results)

predictions = trainer.predict(tokenized_valid)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        pred_labels,
        labels=list(range(len(RELATION_LABELS))),
        target_names=RELATION_LABELS,
        digits=3,
        zero_division=0,
    )
)

{'eval_loss': 0.7275375127792358, 'eval_model_preparation_time': 0.003, 'eval_accuracy': 0.7982062780269058, 'eval_f1': 0.7906044548389858, 'eval_runtime': 1.5881, 'eval_samples_per_second': 140.417, 'eval_steps_per_second': 8.815}
                      precision    recall  f1-score   support

Positive_Correlation      0.757     0.824     0.789        34
Negative_Correlation      0.522     0.571     0.545        21
         Association      0.778     0.648     0.707        54
          Comparison      0.000     0.000     0.000         1
         Cotreatment      0.000     0.000     0.000         1
    Drug_Interaction      0.000     0.000     0.000         0
                Bind      0.000     0.000     0.000         1
          Conversion      0.000     0.000     0.000         0
          NoRelation      0.873     0.928     0.900       111

            accuracy                          0.798       223
           macro avg      0.325     0.330     0.327       223
        weighted avg  

In [28]:
AutoModelForSequenceClassification.from_pretrained(f"arxiv_bert/checkpoint-3550")
 
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
classify = pipeline(task='text-classification', model=model, tokenizer=tokenizer)
 
all_files = glob.glob('arxiv_custom_inference_data/*')
for file_name in all_files:
    file = open(file_name)
    content = file.read()
    print(content)
    result = classify(content)
    print('PRED: ', result)
    print('GT: ', file_name.split('_')[-1].split('.txt')[0])
    print('\n')

OSError: arxiv_bert/checkpoint-3550 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`